In [ ]:
import gym
from gym import spaces
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
class Segment: #(gym.Env):
    def __init__(self,numActions,startObservation,p,terminal,done):
        self.numActions = numActions
        self.observation = startObservation
        self.p = p
        self.terminal = terminal
        self.action_space = spaces.Discrete(self.numActions) # {0:left, 1:right}
        self.observation_space = spaces.Discrete(2*self.terminal+1)
        self.done = done
    def step(self,action):
        assert self.action_space.contains(action)
        assert self.observation_space.contains(self.observation)
        assert self.action_space.n == 2
        assert self.observation!=0
        assert self.observation!=(2*self.terminal)
        observation = self.observation
        done = self.done
        if action==0:
            observation_next = observation + np.random.choice([-1,1],p=[self.p,1-self.p])
        elif action==1:
            observation_next = observation + np.random.choice([-1,1],p=[1-self.p,self.p])
        if observation_next == (2*self.terminal):
            done = True
        elif observation_next == 0:
            done = True
        self.done = done
        self.observation = observation_next
        reward=self.rewards(observation,observation_next,action)
        return [self.observation,reward,done]
    def rewards(self,observation,observation_next,action):
        if observation_next==(2*self.terminal):
            reward = 1.0
        elif observation_next==0:
            reward = -1.0
        else:
            reward = -0.05
        return reward
    def reset(self):
        observation = self.startObservation
        self.observation = observation
        self.done = False
        return observation

In [ ]:
# this is a hard coded policy
def policy(observation):
    action = 1
    return action

In [ ]:
t=0
tMAX = 50
done = False
observation = 10
observations = [observation]
rewards = []
actions = []

In [ ]:
env = Segment(numActions=2,startObservation=10,p=0.8,terminal=10,done=False)
while t <tMAX and done==False:
    action = policy(observation=observation)
    observation_next,reward,done = env.step(action=action)
    observations.append(observation_next)
    rewards.append(reward)
    actions.append(action)
    observation = observation_next
    t+=1

In [ ]:
dta = pd.DataFrame(
             [range(0,t),
              observations[0:t],
              observations[1:(t+1)],
              actions[0:t],
              rewards]).transpose()

dta.columns = ['t','observation','observation_next','action','reward']
dta['observation'] = dta['observation'] - 10
dta['observation_next'] = dta['observation_next'] - 10
dta['beta'] = 0.98
dta['beta^t'] = dta['beta']**dta['t']
dta['beta^t_reward'] = dta['beta^t']*dta['reward']

dta

In [ ]:
allEpisodes = pd.DataFrame()

for e in range(0, 1000):
    t = 0
    tMAX = 50
    done = False
    observation = 10
    observations = [observation]
    rewards = []
    actions = []
    env = Segment(numActions=2, startObservation=10, p=0.8, terminal=10, done=False)
    beta = 0.98  # Discount factor
    beta_t_rewards = []  # To store discounted rewards

    while t < tMAX and not done:
        action = policy(observation=observation)
        observation_next, reward, done = env.step(action=action)
        observations.append(observation_next)
        rewards.append(reward)
        actions.append(action)
        beta_t_rewards.append((beta ** t) * reward)  # Apply discount factor
        observation = observation_next
        t += 1

    # Calculate the total discounted reward for the episode
    total_discounted_reward = sum(beta_t_rewards)

    # Append episode result to the DataFrame
    allEpisodes = pd.concat([
        allEpisodes,
        pd.DataFrame({
            'episode': [e],
            'total_discounted_reward': [total_discounted_reward]
        })
    ], ignore_index=True)

# Calculate summary statistics
summary_stats = allEpisodes['total_discounted_reward'].describe(percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
summary_stats.index = ['count', 'mean', 'std', 'min', '1%', '5%', '10%', '25%', '50%', '75%', '90%', '95%', '99%', 'max']

summary_df = pd.DataFrame(summary_stats)

summary_df

In [ ]:
allEpisodes = pd.DataFrame()

obs = range(1, 20)

for o in obs:
    for e in range(0, 1000):
        t = 0
        tMAX = 50
        done = False
        observation = int(o)
        observations = [observation]
        rewards = []
        actions = []
        env = Segment(numActions=2, startObservation=observation, p=0.8, terminal=10, done=False)
        
        while t < tMAX and not done:
            action = policy(observation=observation)
            observation_next, reward, done = env.step(action=action)
            observations.append(observation_next)
            rewards.append(reward)
            actions.append(action)
            observation = observation_next
            t += 1
        
        dta = pd.DataFrame(
            [[o-10]*t,
            [e]*t,
            range(0, t),
            observations[:t],
            observations[1:(t+1)],
            actions[:t],
            rewards]).transpose()
        
        dta.columns = ['initial', 'episode', 't', 'observation', 'observation_next', 'action', 'reward']
        dta['observation'] = dta['observation'] - 10
        dta['observation_next'] = dta['observation_next'] - 10
        dta['beta'] = 0.98
        dta['beta^t'] = dta['beta']**dta['t']
        dta['beta^t_reward'] = dta['beta^t'] * dta['reward']
        allEpisodes = pd.concat([allEpisodes, dta], ignore_index=True)

In [ ]:
# Compute the sum of discounted rewards for each episode
episode_sums = allEpisodes.groupby(['initial', 'episode'])['beta^t_reward'].sum().reset_index()

# Group by 'initial' and compute summary statistics
summary_stats = episode_sums.groupby('initial')['beta^t_reward'].describe(
    percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]
)

summary_stats

In [ ]:
# Filter for initial state s0 = 0
s0_rewards = episode_sums[episode_sums['initial'] == 0]['beta^t_reward']

plt.figure(figsize=(10, 6))
sns.histplot(s0_rewards, bins = 11, color='blue')

plt.title('Histogram: Sum of Discounted Realized Rewards for $s_0 = 0$', fontsize=16)
plt.xlabel('')
plt.ylabel('Frequency', fontsize=14)

plt.show()

In [ ]:
mean_discounted_rewards = summary_stats['mean']

plt.figure(figsize=(10, 6))
plt.scatter(
    mean_discounted_rewards.index,
    mean_discounted_rewards.values,
    color='black'
)

plt.title('Expected Discounted Rewards vs. Initial States', fontsize=16)
plt.xlabel('Initial State ($s_0$)', fontsize=14)
plt.ylabel('$v(s_0)$', fontsize=14)

plt.xticks(range(-9, 10))

plt.show()

In [ ]:
transition_counts = allEpisodes.groupby(['observation', 'observation_next']).size().reset_index(name='count')

# Normalize counts to probabilities
total_counts = transition_counts.groupby('observation')['count'].transform('sum')
transition_counts['probability'] = transition_counts['count'] / total_counts

transition_matrix = transition_counts.pivot(index='observation', columns='observation_next', values='probability').fillna(0)

transition_matrix

In [ ]:
def left_policy(observation):
    # Always return -1 as the action
    action = 0
    return action

In [ ]:
t=0
tMAX = 50
done = False
observation = 10
observations = [observation]
rewards = []
actions = []

In [ ]:
env = Segment(numActions=2,startObservation=10,p=0.8,terminal=10,done=False)
while t <tMAX and done==False:
    action = left_policy(observation=observation)
    observation_next,reward,done = env.step(action=action)
    observations.append(observation_next)
    rewards.append(reward)
    actions.append(action)
    observation = observation_next
    t+=1

In [ ]:
dta = pd.DataFrame(
             [range(0,t),
              observations[0:t],
              observations[1:(t+1)],
              actions[0:t],
              rewards]).transpose()

dta.columns = ['t','observation','observation_next','action','reward']
dta['observation'] = dta['observation'] - 10
dta['observation_next'] = dta['observation_next'] - 10
dta['beta'] = 0.98
dta['beta^t'] = dta['beta']**dta['t']
dta['beta^t_reward'] = dta['beta^t']*dta['reward']

In [ ]:
dta

In [ ]:
allEpisodes = pd.DataFrame()

for e in range(0, 1000):
    t = 0
    tMAX = 50
    done = False
    observation = 10
    observations = [observation]
    rewards = []
    actions = []
    env = Segment(numActions=2, startObservation=10, p=0.8, terminal=10, done=False)
    beta = 0.98  # Discount factor
    beta_t_rewards = []  # To store discounted rewards

    while t < tMAX and not done:
        action = left_policy(observation=observation)
        observation_next, reward, done = env.step(action=action)
        observations.append(observation_next)
        rewards.append(reward)
        actions.append(action)
        beta_t_rewards.append((beta ** t) * reward)  # Apply discount factor
        observation = observation_next
        t += 1

    # Calculate the total discounted reward for the episode
    total_discounted_reward = sum(beta_t_rewards)

    # Append episode result to the DataFrame
    allEpisodes = pd.concat([
        allEpisodes,
        pd.DataFrame({
            'episode': [e],
            'total_discounted_reward': [total_discounted_reward]
        })
    ], ignore_index=True)

# Calculate summary statistics
summary_stats = allEpisodes['total_discounted_reward'].describe(percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
summary_stats.index = ['count', 'mean', 'std', 'min', '1%', '5%', '10%', '25%', '50%', '75%', '90%', '95%', '99%', 'max']

summary_df = pd.DataFrame(summary_stats)

summary_df

In [ ]:
# Compute the sum of discounted rewards for each episode
episode_sums = allEpisodes.groupby(['initial', 'episode'])['beta^t_reward'].sum().reset_index()

# Group by 'initial' and compute summary statistics
summary_stats = episode_sums.groupby('initial')['beta^t_reward'].describe(
    percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]
)

summary_stats

In [ ]:
# Filter for initial state s0 = 0
s0_rewards = episode_sums[episode_sums['initial'] == 0]['beta^t_reward']

plt.figure(figsize=(10, 6))
sns.histplot(s0_rewards, bins = 11, color='blue')

plt.title('Histogram: Sum of Discounted Realized Rewards for $s_0 = 0$', fontsize=16)
plt.xlabel('')
plt.ylabel('Frequency', fontsize=14)

plt.show()

In [ ]:
mean_discounted_rewards = summary_stats['mean']

plt.figure(figsize=(10, 6))
plt.scatter(
    mean_discounted_rewards.index,
    mean_discounted_rewards.values,
    color='black'
)

plt.title('Expected Discounted Rewards vs. Initial States', fontsize=16)
plt.xlabel('Initial State ($s_0$)', fontsize=14)
plt.ylabel('$v(s_0)$', fontsize=14)

plt.xticks(range(-9, 10))

plt.show()

In [ ]:
transition_counts = allEpisodes.groupby(['observation', 'observation_next']).size().reset_index(name='count')

# Normalize counts to probabilities
total_counts = transition_counts.groupby('observation')['count'].transform('sum')
transition_counts['probability'] = transition_counts['count'] / total_counts

transition_matrix = transition_counts.pivot(index='observation', columns='observation_next', values='probability').fillna(0)

transition_matrix

In [ ]:
import random

def half_policy(observation):
    action = random.choice([0, 1])
    return action

In [ ]:
t=0
tMAX = 50
done = False
observation = 10
observations = [observation]
rewards = []
actions = []

In [ ]:
env = Segment(numActions=2,startObservation=10,p=0.8,terminal=10,done=False)
while t <tMAX and done==False:
    action = half_policy(observation=observation)
    observation_next,reward,done = env.step(action=action)
    observations.append(observation_next)
    rewards.append(reward)
    actions.append(action)
    observation = observation_next
    t+=1

In [ ]:
dta = pd.DataFrame(
             [range(0,t),
              observations[0:t],
              observations[1:(t+1)],
              actions[0:t],
              rewards]).transpose()

dta.columns = ['t','observation','observation_next','action','reward']
dta['observation'] = dta['observation'] - 10
dta['observation_next'] = dta['observation_next'] - 10
dta['beta'] = 0.98
dta['beta^t'] = dta['beta']**dta['t']
dta['beta^t_reward'] = dta['beta^t']*dta['reward']

dta

In [ ]:
allEpisodes = pd.DataFrame()

for e in range(0, 1000):
    t = 0
    tMAX = 50
    done = False
    observation = 10
    observations = [observation]
    rewards = []
    actions = []
    env = Segment(numActions=2, startObservation=10, p=0.8, terminal=10, done=False)
    beta = 0.98  # Discount factor
    beta_t_rewards = []  # To store discounted rewards

    while t < tMAX and not done:
        action = half_policy(observation=observation)
        observation_next, reward, done = env.step(action=action)
        observations.append(observation_next)
        rewards.append(reward)
        actions.append(action)
        beta_t_rewards.append((beta ** t) * reward)  # Apply discount factor
        observation = observation_next
        t += 1

    # Calculate the total discounted reward for the episode
    total_discounted_reward = sum(beta_t_rewards)

    # Append episode result to the DataFrame
    allEpisodes = pd.concat([
        allEpisodes,
        pd.DataFrame({
            'episode': [e],
            'total_discounted_reward': [total_discounted_reward]
        })
    ], ignore_index=True)

# Calculate summary statistics
summary_stats = allEpisodes['total_discounted_reward'].describe(percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
summary_stats.index = ['count', 'mean', 'std', 'min', '1%', '5%', '10%', '25%', '50%', '75%', '90%', '95%', '99%', 'max']

summary_df = pd.DataFrame(summary_stats)

summary_df

In [ ]:
allEpisodes = pd.DataFrame()

obs = range(1, 20)

for o in obs:
    for e in range(0, 1000):
        t = 0
        tMAX = 50
        done = False
        observation = int(o)
        observations = [observation]
        rewards = []
        actions = []
        env = Segment(numActions=2, startObservation=observation, p=0.8, terminal=10, done=False)
        
        while t < tMAX and not done:
            action = half_policy(observation=observation)
            observation_next, reward, done = env.step(action=action)
            observations.append(observation_next)
            rewards.append(reward)
            actions.append(action)
            observation = observation_next
            t += 1
        
        dta = pd.DataFrame(
            [[o-10]*t,
            [e]*t,
            range(0, t),
            observations[:t],
            observations[1:(t+1)],
            actions[:t],
            rewards]).transpose()
        
        dta.columns = ['initial', 'episode', 't', 'observation', 'observation_next', 'action', 'reward']
        dta['observation'] = dta['observation'] - 10
        dta['observation_next'] = dta['observation_next'] - 10
        dta['beta'] = 0.98
        dta['beta^t'] = dta['beta']**dta['t']
        dta['beta^t_reward'] = dta['beta^t'] * dta['reward']
        allEpisodes = pd.concat([allEpisodes, dta], ignore_index=True)

In [ ]:
# Compute the sum of discounted rewards for each episode
episode_sums = allEpisodes.groupby(['initial', 'episode'])['beta^t_reward'].sum().reset_index()

# Group by 'initial' and compute summary statistics
summary_stats = episode_sums.groupby('initial')['beta^t_reward'].describe(
    percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]
)

summary_stats

In [ ]:
# Filter for initial state s0 = 0
s0_rewards = episode_sums[episode_sums['initial'] == 0]['beta^t_reward']

plt.figure(figsize=(10, 6))
sns.histplot(s0_rewards, bins = 11, color='blue')

plt.title('Histogram: Sum of Discounted Realized Rewards for $s_0 = 0$', fontsize=16)
plt.xlabel('')
plt.ylabel('Frequency', fontsize=14)

plt.show()

In [ ]:
mean_discounted_rewards = summary_stats['mean']

plt.figure(figsize=(10, 6))
plt.scatter(
    mean_discounted_rewards.index,
    mean_discounted_rewards.values,
    color='black'
)

plt.title('Expected Discounted Rewards vs. Initial States', fontsize=16)
plt.xlabel('Initial State ($s_0$)', fontsize=14)
plt.ylabel('$v(s_0)$', fontsize=14)

plt.xticks(range(-9, 10))

plt.show()